In [1]:
import tensorflow as tf
import TensorSlider as ts
import keras


2025-02-06 18:21:47.420985: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1738862507.442897   72030 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1738862507.450411   72030 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-02-06 18:21:47.478622: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [3]:
def createLabelsBatch(data, lookforward):
    """
    create input labels from the lookahead data
    """

    shape = lookforward.shape

    dividepricesby = tf.reshape(lookforward[:,0,0,0], (shape[0], 1, 1, 1))
    prices = tf.divide(lookforward[:,:,0:4], dividepricesby) # divide by latest base timeframe close
    prices -= 1 # zero out
    prices *= 10 # convert to 1/10 percentage, so 1 = 10 percent

    low = tf.reduce_min(prices[:,0,2], 1)
    high = tf.reduce_max(prices[:,0,1], 1)

    label = tf.stack([low, high], axis = 1)



    shape = data.shape
    # data processing, format: batch, timeframe, feature(ohlcv), window (ascending time)
    # prices
    dividepricesby = tf.reshape(data[:,0,3,-1], (shape[0], 1, 1, 1))
    prices = tf.divide(data[:,:,0:4], dividepricesby) # divide by latest base timeframe close
    prices -= 1 # zero out
    prices *= 10 # convert to 1/10 percentage, so 1 = 10 percent
    
    volumes = tf.reshape(data[:,:,4], (shape[0], shape[1], 1, shape[3]))
    dividebyvolume = tf.reshape(data[:,:,4,-1], (shape[0], shape[1], 1, 1))
    volume = tf.divide(volumes, dividebyvolume) # divide by latest volume (of each timeframe)
    volume -= 1
    volume *= 10 # convert to 1/10 percentage, so 1 = 10 percent
    
    data = tf.clip_by_value(tf.concat([prices, volume], axis=2), -2, 2)
    # remove nans
    data, label = tf.keras.ops.nan_to_num(data), tf.keras.ops.nan_to_num(label)

    return data, label

def decode(record_bytes):
    # Function for parsing each record in the tf files
    example = tf.io.parse_single_example(
        # Data
        record_bytes,

        # Schema
        {
        'Timeframe': tf.io.FixedLenFeature([], tf.string),
        'timestamp': tf.io.RaggedFeature(dtype=tf.int64),
        'Open': tf.io.RaggedFeature(dtype=tf.float32),
        'High': tf.io.RaggedFeature(dtype=tf.float32),
        'Low': tf.io.RaggedFeature(dtype=tf.float32),
        'Close': tf.io.RaggedFeature(dtype=tf.float32),
        'Volume': tf.io.RaggedFeature(dtype=tf.float32),
        }
        )

    return example

def getDataset(path):
    ds = tf.data.TFRecordDataset(path,  num_parallel_reads = tf.data.AUTOTUNE)
    ds = ds.map(decode, num_parallel_calls = tf.data.AUTOTUNE)
    return ds

## Get Datasets

In [6]:
tfrecordpath = "../Data/tfrecords/"

windowsize = 100
lookahead = 5
batch_size = 100

def getSlider(coin):

    path = tfrecordpath + coin+"/{tframe}.tfrecord"
    datasets = {"1m" : getDataset(path.format(tframe="1m")),
                "5m":  getDataset(path.format(tframe="5m")),
                "15m":  getDataset(path.format(tframe="15m")),
                "30m":  getDataset(path.format(tframe="30m")),
                "1h":  getDataset(path.format(tframe="1h"))}
    return ts.WindowSlider(datasets, windowsize, lookahead, batch_size)

def getSliders(coins):
    datasets = []
    for coin in coins:
        datasets.append(tf.data.Dataset.from_generator(lambda: getSlider(coin),
            output_signature=(
                tf.TensorSpec((batch_size,5,5,windowsize), dtype=tf.float32),
                tf.TensorSpec((batch_size,5,5, lookahead+1), dtype=tf.float32))
            ))#.prefetch(tf.data.AUTOTUNE))
    return datasets

coins = ["BTCUSD_PERP", "ETHUSD_PERP", "ADAUSD_PERP", "SOLUSD_PERP"]
datasets = getSliders(coins)

## Combine Datasets

In [7]:
#ds = tf.data.Dataset.sample_from_datasets(datasets = datasets).prefetch(tf.data.AUTOTUNE)
ds = datasets[0]

ds = ds.map(createLabelsBatch, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

In [8]:
import keras
import os

def load_model(name):
    """
    Load model and latest checkpoint (if applicable). Returns model, and last epoch that was trained.
    If no checkpoints present, either creates the checkpoint folder or trains directly on the saved model.
    """
    folder = "models/" + name + "/"
    # Load model
    model = keras.models.load_model(folder + "model.keras")

    # Check if we have any checkpoints in the first place
    if not os.path.exists(folder + "checkpoints"):
        # No checkpoint folder, lets create one, and return the model, epoch 0
        os.mkdir(folder + "checkpoints")
        return model, 0

    # Check how many epoch checkpoints we have in the (existing!) checkpoint folder
    checkpoints = os.listdir("models/" + name + "/checkpoints/")

    # We do this by just counting how many files are in there, we assume there will be no vandalism
    # and all files inside the checkpoint folder are checkpoints
    lastEpoch = len(checkpoints)

    # load the latest checkpoint if we have more than 1 of them
    if lastEpoch >= 1:
        model.load_weights("models/" + name + "/checkpoints/" + str(lastEpoch) + ".weights.h5")

    return model, lastEpoch

class saveEachEpoch(tf.keras.callbacks.Callback):
    """
    custom callback for saving checkpoints because of course we need to do this on our own
    """

    def __init__(self, name, lastEpoch=0):
        super().__init__()
        # no idea if we want to or need to super this
        self.lastEpoch = lastEpoch
        self.name = name
        print("Model was trained for " + str(lastEpoch) + " epochs before.")

    def on_epoch_end(self, epoch, logs):
        """
        Create a checkpoint and save.
        """
        # Increment which epoch this is
        self.lastEpoch += 1
        # Save weights
        self.model.save_weights("models/" + str(self.name) + "/checkpoints/" + str(self.lastEpoch) + ".weights.h5")
        print("\nSaved checkpoint of epoch number " + str(self.lastEpoch))
        # Save the latest model
        self.model.save("models/" + str(self.name) + "/model.keras")


In [9]:
modelName = "modeltest"

tensorboard = keras.callbacks.TensorBoard(
                                                log_dir=f"models/{modelName}/logs",
                                                histogram_freq=1,
                                                write_graph=True,
                                                write_images=False,
                                                write_steps_per_second=True,
                                                update_freq="batch",
                                                profile_batch = '70,100',
                                                embeddings_freq=0,
                                                embeddings_metadata=None,
                                            )

model, epochs = load_model(modelName)

history = model.fit(ds, epochs=10, verbose=1, validation_data=None, callbacks=[saveEachEpoch(modelName, epochs), tensorboard])

2025-02-06 18:22:08.541340: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:103] Profiler session initializing.
2025-02-06 18:22:08.541390: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:118] Profiler session started.
2025-02-06 18:22:08.542196: I external/local_xla/xla/backends/profiler/gpu/cupti_tracer.cc:1006] Profiler found 1 GPUs
2025-02-06 18:22:08.583272: W external/local_xla/xla/backends/profiler/gpu/cupti_tracer.cc:1177] Fail to use per-thread activity buffer, cupti trace overhead may be big. CUPTI ERROR CODE:1
2025-02-06 18:22:08.583460: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:130] Profiler session tear down.
2025-02-06 18:22:08.583643: I external/local_xla/xla/backends/profiler/gpu/cupti_tracer.cc:1213] CUPTI activity buffer flushed


Model was trained for 0 epochs before.
Epoch 1/10


/root/miniconda3/envs/levbot/lib/python3.11/site-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 34 variables whereas the saved optimizer has 2 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))
2025-02-06 18:22:10.458943: I tensorflow/core/kernels/data/tf_record_dataset_op.cc:370] TFRecordDataset `buffer_size` is unspecified, default to 262144
I0000 00:00:1738862540.466796   72092 service.cc:148] XLA service 0x7f4e8c010180 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1738862540.466958   72092 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 3060 Ti, Compute Capability 8.6
2025-02-06 18:22:20.540030: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1738862540.810100   72092 cuda_dnn.cc:529] Loaded cuDNN ve

     14/Unknown 20s 9ms/step - MeanAbsolutePercentageError: 16968318.0000 - MeanSquaredError: 1.3653 - loss: 1.3653

I0000 00:00:1738862549.622549   72092 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


...

    114/Unknown 21s 10ms/step - MeanAbsolutePercentageError: 7175066.0000 - MeanSquaredError: 0.3425 - loss: 0.3425

2025-02-06 18:22:30.700689: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:68] Profiler session collecting data.
2025-02-06 18:22:30.706918: I external/local_xla/xla/backends/profiler/gpu/cupti_tracer.cc:1213] CUPTI activity buffer flushed
2025-02-06 18:22:30.720771: I external/local_xla/xla/backends/profiler/gpu/cupti_collector.cc:635]  GpuTracer has collected 3949 callback api events and 3612 activity events. 
2025-02-06 18:22:30.720801: I external/local_xla/xla/backends/profiler/gpu/cupti_collector.cc:638]  GpuTracer max callback_events: 2097152, max activity events: 2097152
2025-02-06 18:22:30.727611: I external/local_tsl/tsl/profiler/lib/profiler_session.cc:130] Profiler session tear down.
2025-02-06 18:22:30.747949: I external/local_xla/xla/tsl/profiler/rpc/client/save_profile.cc:147] Collecting XSpace to repository: models/modeltest/logs/train/plugins/profile/2025_02_06_18_22_30/Alex-Desktop.xplane.pb


   3340/Unknown 47s 8ms/step - MeanAbsolutePercentageError: 2361361.0000 - MeanSquaredError: 0.0232 - loss: 0.0232

2025-02-06 18:22:56.559914: W tensorflow/core/framework/op_kernel.cc:1829] INVALID_ARGUMENT: ValueError: cannot assign slice of shape (5, 2) from input of shape (5, 6)
Traceback (most recent call last):

  File "/root/miniconda3/envs/levbot/lib/python3.11/site-packages/tensorflow/python/ops/script_ops.py", line 269, in __call__
    ret = func(*args)
          ^^^^^^^^^^^

  File "/root/miniconda3/envs/levbot/lib/python3.11/site-packages/tensorflow/python/autograph/impl/api.py", line 643, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^

  File "/root/miniconda3/envs/levbot/lib/python3.11/site-packages/tensorflow/python/data/ops/from_generator_op.py", line 198, in generator_py_func
    values = next(generator_state.get_iterator(iterator_id))
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

  File "/mnt/c/Users/alexs/Desktop/levbot/Training/TensorSlider.py", line 71, in __next__
    return self.speedslider.next()
           ^^^^^^^^^^^^^^^^^^^^^^^

InvalidArgumentError: Graph execution error:

Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
Detected at node PyFunc defined at (most recent call last):
<stack traces unavailable>
2 root error(s) found.
  (0) INVALID_ARGUMENT:  ValueError: cannot assign slice of shape (5, 2) from input of shape (5, 6)
Traceback (most recent call last):

  File "/root/miniconda3/envs/levbot/lib/python3.11/site-packages/tensorflow/python/ops/script_ops.py", line 269, in __call__
    ret = func(*args)
          ^^^^^^^^^^^

  File "/root/miniconda3/envs/levbot/lib/python3.11/site-packages/tensorflow/python/autograph/impl/api.py", line 643, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^

  File "/root/miniconda3/envs/levbot/lib/python3.11/site-packages/tensorflow/python/data/ops/from_generator_op.py", line 198, in generator_py_func
    values = next(generator_state.get_iterator(iterator_id))
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^